In [13]:
tasks = [
    "bace",
    "smol-property_prediction-bbbp",
    "smol-property_prediction-clintox",
    "smol-property_prediction-esol",
    "smol-property_prediction-lipo",
    "smol-property_prediction-hiv",
    "smol-property_prediction-sider",
    "qm9_homo",
    "qm9_lumo",
    "qm9_homo_lumo_gap",
    "chebi-20-text2mol",
    "chebi-20-mol2text",
    "smol-molecule_generation",
    "smol-molecule_captioning",
    "reagent_prediction",
    "forward_reaction_prediction",
    "smol-forward_synthesis",
    "retrosynthesis",
    "smol-retrosynthesis",
]


string_only_paths = {}
# classification
string_only_paths['bace'] = '/data/all_checkpoints/bace_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['bbbp'] = '/data/all_checkpoints/smol-property_prediction-bbbp_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['clintox'] = '/data/all_checkpoints/smol-property_prediction-clintox_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['hiv'] = '/data/all_checkpoints/smol-property_prediction-hiv_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['sider'] = '/data/all_checkpoints/smol-property_prediction-sider_mlp_string_only_12ep_0318/lightning_logs/version_0'

# regression
string_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_mlp_string_only_12ep_0318/lightning_logs/version_0'

# captioning
string_only_paths['chebi-20-mol2text'] = '/data/all_checkpoints/chebi-20-mol2text_mlp_string_only_12ep_0318/lightning_logs/version_0'

# reaction
string_only_paths['forward'] = '/data/all_checkpoints/forward_reaction_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'
string_only_paths['reagent'] = '/data/all_checkpoints/reagent_prediction_mlp_string_only_12ep_0318/lightning_logs/version_0'


graph_only_paths = {}
graph_only_paths['esol'] = '/data/all_checkpoints/smol-property_prediction-esol_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['lipo'] = '/data/all_checkpoints/smol-property_prediction-lipo_qformer_graph_only_12ep_0318/lightning_logs/version_0'
graph_only_paths['qm9_homo'] = '/data/all_checkpoints/qm9_homo_qformer_graph_only_12ep_0318/lightning_logs/version_0'

In [43]:
import os
import json
import re
import selfies as sf
from rdkit import Chem

def get_prediction_dump(path):
    prediction_files = [f for f in os.listdir(path) if f.startswith('ft-') and f.endswith('.json')]
    output_files = [f for f in prediction_files if 'output' in f]

    # read the output files
    output_data = []
    for p in output_files:
        with open(os.path.join(path, p), 'r') as f:
            data = json.load(f)
            output_data.extend(data)

    processed_data = []
    truncated_idx = []

    for i in range(len(output_data)):
        instance = output_data[i]
        prediction = instance['prediction']
        target = instance['target']
        prompt = instance['prompt']
        selfies_pattern = r"(?<=<SELFIES>).*(?=</SELFIES>)"
        try:
            selfies_string = re.search(selfies_pattern, prompt).group().replace(" ", "")
            out = {
                'selfies' : selfies_string,
                'prediction': prediction,
                'target': target
            }
            if 'prob' in instance.keys():
                out['prob'] = instance['prob']
            processed_data.append(out)
        except:
            truncated_idx.append(i)
    # print the truncated ratio
    print(f"Truncated ratio: {len(truncated_idx) / len(output_data)}")
    return processed_data

def convert_string2number(text):
    text = text.replace("<FLOAT>", "").replace("</FLOAT>", "")
    text = text.replace("<", "").replace(">", "").replace("|", "").replace(" ", "").replace("/s", "")
    return float(text)

def rank_error_regression(data):
    mae = []
    for i in range(len(data)):
        prediciton = convert_string2number(data[i]['prediction'])
        target = convert_string2number(data[i]['target'])
        data[i]['prediction'] = prediciton
        data[i]['target'] = target
        mae.append(abs(prediciton - target))
    #  sort the data by mae, and add mae rank to the data
    sorted_data = [x for _, x in sorted(zip(mae, data), key=lambda pair: pair[0])]
    for i in range(len(sorted_data)):
        sorted_data[i]['error_rank'] = i + 1
    return sorted_data

In [44]:
string_only_esol_dump = get_prediction_dump(string_only_paths['esol'])
string_only_esol = rank_error_regression(string_only_esol_dump)

string_only_lipo_dump = get_prediction_dump(string_only_paths['lipo'])
string_only_lipo = rank_error_regression(string_only_lipo_dump)

string_only_qm9_homo_dump = get_prediction_dump(string_only_paths['qm9_homo'])
string_only_qm9_homo = rank_error_regression(string_only_qm9_homo_dump)

Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0


In [50]:
string_only_qm9_homo
list_smiles = [sf.decoder(i['selfies']) for i in string_only_qm9_homo]
list_mol = [Chem.MolFromSmiles(i) for i in list_smiles]
list_rank = [i['error_rank'] for i in string_only_qm9_homo]
list_mae = [abs(i['prediction'] - i['target']) for i in string_only_qm9_homo]

In [ ]:
# line plot by erro

In [49]:
len(list_mol)

688

In [48]:
mol = list_mol[0]
mol.GetNumAtoms()

9

In [ ]:
# complete bace, bbbp, clintox, hiv, sider, chebi-20-mol2text, forward, reagent
string_only_bace = get_prediction_dump(string_only_paths['bace'])
string_only_bbbp = get_prediction_dump(string_only_paths['bbbp'])
string_only_clintox = get_prediction_dump(string_only_paths['clintox'])
string_only_hiv = get_prediction_dump(string_only_paths['hiv'])
string_only_sider = get_prediction_dump(string_only_paths['sider'])

string_only_esol_dump = get_prediction_dump(string_only_paths['esol'])
string_only_esol = rank_error_regression(string_only_esol_dump)

string_only_lipo_dump = get_prediction_dump(string_only_paths['lipo'])
string_only_lipo = rank_error_regression(string_only_lipo_dump)

string_only_qm9_homo_dump = get_prediction_dump(string_only_paths['qm9_homo'])
string_only_qm9_homo = rank_error_regression(string_only_qm9_homo_dump)

string_only_chebi_20_mol2text = get_prediction_dump(string_only_paths['chebi-20-mol2text'])

string_only_forward = get_prediction_dump(string_only_paths['forward'])
string_only_reagent = get_prediction_dump(string_only_paths['reagent'])




Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.034916201117318434
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.0
Truncated ratio: 0.001513317191283293
Truncated ratio: 0.0
Truncated ratio: 0.0


In [18]:
string_only_esol[0]

{'selfies': '[C][C][C][C][N][C][=Branch1][C][=O][N][C][Branch1][Branch2][N][C][=Branch1][C][=O][O][C][=N][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][=C]',
 'prediction': '<FLOAT> <|-|> <|2|> <|.|> <|8|> <|6|> <|3|> <|0|> </FLOAT> </s>',
 'target': '<FLOAT> <|-|> <|4|> <|.|> <|8|> <|8|> <|3|> <|0|> </FLOAT> </s>'}

In [17]:
def convert_string2number(text):
    text = text.replace("<FLOAT>", "").replace("</FLOAT>", "")
    text = text.replace("<", "").replace(">", "").replace("|", "").replace(" ", "").replace("/s", "")
    return float(text)

instance = string_only_esol[0]
prediction_number = convert_string2number(instance['prediction'])
target_number = convert_string2number(instance['target'])
prediction_number, target_number

(-2.863, -4.883)

In [ ]:
instance = esol_prediction[0]


prediction = instance['prediction']
target = instance['target']
prompt = instance['prompt']
selfies_pattern = r"(?<=<SELFIES>).*(?=</SELFIES>)"
selfies_string = re.search(selfies_pattern, prompt).group().replace(" ", "")
out = {
    'selfies' : selfies_string,
    'prediction': prediction,
    'target': target
}

In [ ]:
selfies_string

'[C][C][C][C][N][C][=Branch1][C][=O][N][C][Branch1][Branch2][N][C][=Branch1][C][=O][O][C][=N][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][=C]'